In [1]:
# Import required libraries
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from typing import TypedDict

# Load environment variables
from dotenv import load_dotenv
import os

# Gemini LLM import
from langchain_google_genai import ChatGoogleGenerativeAI

In [2]:
# Load .env file (IMPORTANT)
load_dotenv()

# Get API key from .env
api_key = os.getenv("GOOGLE_API_KEY")


python-dotenv could not parse statement starting at line 2


In [8]:
# Initialize Gemini model
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=api_key
)


# Define state structure
class ChatState(TypedDict):
    message: str
    response: str

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


In [9]:
# Define chatbot node (Gemini version)
def chatbot(state: ChatState):
    user_msg = state["message"]

    # Call Gemini LLM
    result = llm.invoke(user_msg)

    return {"response": result.content}


In [10]:
# Create graph
graph = StateGraph(ChatState)

# Add node
graph.add_node("chatbot", chatbot)

# Define edges
graph.add_edge(START, "chatbot")
graph.add_edge("chatbot", END)


# Add persistence (memory)
memory = MemorySaver()


In [11]:
# Compile graph with persistence
workflow = graph.compile(checkpointer=memory)


# Thread ID (IMPORTANT for persistence)
config = {"configurable": {"thread_id": "user1"}}

In [12]:
# First run
result1 = workflow.invoke({"message": "Hello"}, config=config)
print("Response 1:", result1["response"])


# Second run (same thread → memory continues)
result2 = workflow.invoke({"message": "Tell me more"}, config=config)
print("Response 2:", result2["response"])

Response 1: Hello! How can I help you today?
Response 2: I'd love to! But "Tell me more" is quite an open invitation. To give you the best and most helpful response, I need a little more context.

What would you like me to tell you more about? For example, are you interested in:

*   **A specific topic?** (e.g., "Tell me more about black holes," "Tell me more about the history of the internet," "Tell me more about a specific animal.")
*   **Something we were just discussing?** (If this is a continuation of a previous conversation.)
*   **A particular type of information?** (e.g., "Tell me a story," "Give me some interesting facts," "Explain a concept to me.")
*   **A specific field?** (e.g., "Tell me more about science," "Tell me more about art," "Tell me more about technology.")

Just give me a hint, and I'll do my best to elaborate!
